In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%%time
!pip install -q pandas numpy scikit-learn xgboost lightgbm \
    rapidfuzz jellyfish \
    sentence-transformers faiss-cpu \
    transformers accelerate datasets \
    torch tqdm unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 1.2 MB/s eta 0:00:00
CPU times: user 475 ms, sys: 197 ms, total: 672 ms
Wall time: 23.4 s


In [ ]:
# Enable 10GB extra virtual RAM on Colab
!fallocate -l 10G /swapfile
!chmod 600 /swapfile
!mkswap /swapfile
!swapon /swapfile
!free -h


Setting up swapspace version 1, size = 10 GiB (10737414144 bytes)
no label, UUID=3c53b633-1779-4e4f-bc40-378d75262991
swapon: /swapfile: swapon failed: Invalid argument
               total        used        free      shared  buff/cache   available
Mem:            47Gi       2.0Gi        31Gi       5.0Mi        14Gi        45Gi
Swap:             0B          0B          0B


In [ ]:
import os, re, gc, json, warnings, time
from collections import defaultdict
from pathlib import Path

import pandas as pd
import numpy as np

from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, JaroWinkler
import jellyfish
from unidecode import unidecode

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupShuffleSplit
import scipy.sparse as sp

import xgboost as xgb
import lightgbm as lgb

from tqdm.auto import tqdm
tqdm.pandas()

import torch

# Device and VRAM check (fixed 'total_mem' -> 'total_memory')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)

# ============================================================
# PATHS — pointing to your mounted Google Drive directory
# ============================================================
BASE_DIR = Path('/content/drive/MyDrive/amazon/student_resource')
TRAIN_DIR = BASE_DIR / 'dataset' / 'train'
TEST_DIR  = BASE_DIR / 'dataset' / 'test'

# Local output directory on Colab runtime for fast read/write
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Verify path connection
print(f"Drive path exists: {BASE_DIR.exists()}")
if BASE_DIR.exists():
    print("Contents:", [p.name for p in BASE_DIR.iterdir()])

# ============================================================
# HYPERPARAMETERS — tune these
# ============================================================
TFIDF_TOP_K        = 25      # candidates per S1 entity from TF-IDF
SBERT_TOP_K        = 25      # candidates per S1 entity from SBERT
SBERT_MIN_SIM      = 0.25    # min cosine sim for SBERT candidates
TFIDF_MIN_SIM      = 0.08    # min cosine sim for TF-IDF candidates
RULE_MIN_TOKENS    = 2       # min shared tokens for rule-based blocking
VAL_SPLIT          = 0.15    # validation fraction
NEG_RATIO          = 5       # hard negatives per positive
XGB_THRESHOLD      = 0.5     # will be optimized later
BATCH_SIZE_SBERT   = 512     # reduce if OOM
BATCH_SIZE_CROSS   = 32      # reduce if OOM

print("✅ Imports and config done")

Device: cpu
Drive path exists: True
Contents: ['Documentation_template.md', '.DS_Store', 'README.md', 'dataset', 'utils']
✅ Imports and config done


In [ ]:
import os
import pandas as pd

SAVE_DIR = '/content/drive/MyDrive/amazon/amazon_ml_challenge/'

print("⚡ Loading preprocessed data from Google Drive in 3 seconds...")
train_s1 = pd.read_parquet(os.path.join(SAVE_DIR, 'train_s1.parquet'))
train_s23 = pd.read_parquet(os.path.join(SAVE_DIR, 'train_s23.parquet'))

# Restore index lookups
train_s1.set_index('entity_id', drop=False, inplace=True)
train_s23.set_index('entity_id', drop=False, inplace=True)
train_s1_lookup = train_s1
train_s23_lookup = train_s23

print(f"✅ Loaded {len(train_s1):,} S1 rows and {len(train_s23):,} S2+S3 rows! Ready to run Cell 9.")


⚡ Loading preprocessed data from Google Drive in 3 seconds...
✅ Loaded 2,206,821 S1 rows and 10,320,219 S2+S3 rows! Ready to run Cell 9.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Automatically find and load train_ground_truth.tsv
if 'train_gt' not in globals() or train_gt is None:
    gt_path = None
    possible_paths = [
        Path('/content/drive/MyDrive/amazon/student_resource/dataset/train/train_ground_truth.tsv'),
        Path('/content/drive/MyDrive/amazon_ml_challenge/train_ground_truth.tsv'),
        Path('/content/student_resource/dataset/train/train_ground_truth.tsv'),
    ]
    for p in possible_paths:
        if p.exists():
            gt_path = p
            break

    if gt_path is None:
        # Search /content for the file
        for root, dirs, files in os.walk('/content'):
            if 'train_ground_truth.tsv' in files:
                gt_path = Path(root) / 'train_ground_truth.tsv'
                break

    print(f"Loading ground truth from: {gt_path}")
    train_gt = pd.read_csv(gt_path, sep='\t')
    print(f"✅ Loaded train_gt: {len(train_gt):,} rows")

# 2. Train / Validation Split
VAL_SPLIT = 0.15

def parse_ground_truth(gt_df):
    gt_dict = {}
    for _, row in gt_df.iterrows():
        s1_id = row['source1_entity_id']
        val = row['matched_entity_ids']
        if pd.isna(val) or not str(val).strip():
            gt_dict[s1_id] = set()
        else:
            gt_dict[s1_id] = set(m.strip() for m in str(val).split(',') if m.strip())
    return gt_dict

full_gt = parse_ground_truth(train_gt)
all_s1_ids = list(full_gt.keys())
np.random.seed(42)
np.random.shuffle(all_s1_ids)

split_idx = int(len(all_s1_ids) * (1 - VAL_SPLIT))
train_s1_ids = set(all_s1_ids[:split_idx])
val_s1_ids   = set(all_s1_ids[split_idx:])

train_gt_dict = {k: v for k, v in full_gt.items() if k in train_s1_ids}
val_gt_dict   = {k: v for k, v in full_gt.items() if k in val_s1_ids}

print(f"\n✅ Split complete!")
print(f"   Train S1: {len(train_gt_dict):,}")
print(f"   Val   S1: {len(val_gt_dict):,} (val_s1_ids is now ready!)")


Loading ground truth from: /content/drive/MyDrive/amazon/student_resource/dataset/train/train_ground_truth.tsv
✅ Loaded train_gt: 2,206,821 rows

✅ Split complete!
   Train S1: 1,875,797
   Val   S1: 331,024 (val_s1_ids is now ready!)


In [ ]:
def compute_f05_macro(predictions: dict, ground_truth: dict) -> dict:
    """
    Compute macro-averaged F0.5 across all S1 entities.
    predictions:   {s1_id: set(predicted_match_ids)}
    ground_truth:  {s1_id: set(true_match_ids)}
    Returns dict with f05, precision, recall, and per-entity scores.
    """
    f05_scores = []
    precisions = []
    recalls = []

    for s1_id in ground_truth:
        true = ground_truth[s1_id]
        pred = predictions.get(s1_id, set())
        if isinstance(pred, list):
            pred = set(pred)
        if isinstance(true, list):
            true = set(true)

        if len(pred) == 0 and len(true) == 0:
            f05_scores.append(1.0)
            precisions.append(1.0)
            recalls.append(1.0)
            continue
        if len(pred) == 0:
            f05_scores.append(0.0)
            precisions.append(0.0)
            recalls.append(0.0)
            continue
        if len(true) == 0:
            f05_scores.append(0.0)
            precisions.append(0.0)
            recalls.append(1.0)
            continue

        tp = len(pred & true)
        fp = len(pred - true)
        fn = len(true - pred)

        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0

        if p + r > 0:
            f05 = (1.25 * p * r) / (0.25 * p + r)
        else:
            f05 = 0.0

        f05_scores.append(f05)
        precisions.append(p)
        recalls.append(r)

    return {
        'f05': np.mean(f05_scores),
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'n_entities': len(f05_scores),
    }

print("✅ F0.5 evaluator ready")

# Quick test
test_pred = {'A': {'X', 'Y'}, 'B': set()}
test_true = {'A': {'X', 'Z'}, 'B': set()}
print("Test:", compute_f05_macro(test_pred, test_true))

✅ F0.5 evaluator ready
Test: {'f05': np.float64(0.75), 'precision': np.float64(0.75), 'recall': np.float64(0.75), 'n_entities': 2}


In [ ]:
%%time
import gc
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

def tfidf_blocking(s1_df, s23_df, top_k=25, min_sim=0.08, batch_size=100):
    """
    Ultra-lean, pure sparse TF-IDF blocking for Kaggle.
    - NEVER calls .toarray() (eliminates 5 GB dense memory spikes!)
    - Fits vocabulary on a sample in 5 seconds
    - Uses only ~15 MB per batch instead of 5,000 MB
    """
    candidates = {eid: set() for eid in s1_df['entity_id']}
    countries = s1_df['country'].dropna().unique()

    for c in countries:
        s1_c = s1_df[s1_df['country'] == c]
        s23_c = s23_df[s23_df['country'] == c]
        if len(s1_c) == 0 or len(s23_c) == 0:
            continue

        print(f"\n  [Country: {c}] S1: {len(s1_c):,} | S2+S3: {len(s23_c):,}")

        # 1. Compact 3-4 char n-gram vectorizer (cuts matrix memory in half)
        vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 4),
            min_df=5,
            max_features=75000,
            sublinear_tf=True,
            dtype=np.float32
        )

        # 2. Fit vocabulary on a sample in 5 seconds (avoids 10 GB memory spike)
        sample_size = min(300000, len(s23_c))
        print(f"    Fitting vocabulary on {sample_size:,} sample texts...")
        sample_texts = s23_c['combined_text'].sample(sample_size, random_state=42)
        vectorizer.fit(sample_texts)
        del sample_texts
        gc.collect()

        # 3. Transform into compact sparse CSR matrices
        print(f"    Transforming matrices...")
        s23_matrix = vectorizer.transform(s23_c['combined_text'])
        s1_matrix  = vectorizer.transform(s1_c['combined_text'])

        s1_ids  = s1_c['entity_id'].values
        s23_ids = s23_c['entity_id'].values

        # 4. Pure Sparse Dot Product (Never converts to dense!)
        for start in tqdm(range(0, len(s1_ids), batch_size), desc=f"    Sparse TF-IDF ({c})"):
            end = min(start + batch_size, len(s1_ids))
            batch_mat = s1_matrix[start:end]

            # Sparse matrix product: shape (batch_size, n_s23) -> stays SPARSE!
            sim_sparse = batch_mat @ s23_matrix.T

            for j in range(sim_sparse.shape[0]):
                row = sim_sparse.getrow(j)
                data = row.data
                indices = row.indices

                if len(data) == 0:
                    continue

                mask = data >= min_sim
                v_data = data[mask]
                v_idx  = indices[mask]

                if len(v_data) > top_k:
                    top_sub = np.argpartition(v_data, -top_k)[-top_k:]
                    selected_ids = v_idx[top_sub]
                else:
                    selected_ids = v_idx

                s1_id = s1_ids[start + j]
                candidates[s1_id] = set(s23_ids[selected_ids])

            del sim_sparse

        del s23_matrix, s1_matrix, vectorizer
        gc.collect()

    total_cands = sum(len(v) for v in candidates.values())
    avg_cands = total_cands / max(1, len(candidates))
    print(f"\n  ✅ TF-IDF blocking complete: {total_cands:,} total candidates (avg {avg_cands:.1f}/entity)")
    return candidates

# Validation sample of 5,000 for quick recall evaluation
val_s1_sample = train_s1[train_s1['entity_id'].isin(val_s1_ids)].sample(min(5000, len(val_s1_ids)), random_state=42)
tfidf_cands_val = tfidf_blocking(val_s1_sample, train_s23, top_k=TFIDF_TOP_K, min_sim=TFIDF_MIN_SIM, batch_size=100)



  [Country: US] S1: 3,030 | S2+S3: 6,186,873
    Fitting vocabulary on 300,000 sample texts...
    Transforming matrices...


    Sparse TF-IDF (US):   0%|          | 0/31 [00:00<?, ?it/s]


  [Country: India] S1: 1,970 | S2+S3: 4,133,346
    Fitting vocabulary on 300,000 sample texts...
    Transforming matrices...


    Sparse TF-IDF (India):   0%|          | 0/20 [00:00<?, ?it/s]


  ✅ TF-IDF blocking complete: 125,000 total candidates (avg 25.0/entity)
CPU times: user 29min 6s, sys: 45.3 s, total: 29min 51s
Wall time: 29min 52s


In [ ]:
import os
import pickle

SAVE_DIR = '/content/drive/MyDrive/amazon_ml_challenge/'
os.makedirs(SAVE_DIR, exist_ok=True)

print("💾 Saving Cell 9 results to Google Drive...")

# 1. Save the candidates dictionary
with open(os.path.join(SAVE_DIR, 'tfidf_cands_val.pkl'), 'wb') as f:
    pickle.dump(tfidf_cands_val, f, protocol=pickle.HIGHEST_PROTOCOL)

# 2. Make sure val_s1_df is set for downstream cells and save to Drive
if 'val_s1_df' not in globals() and 'val_s1_sample' in globals():
    val_s1_df = val_s1_sample

val_s1_df.to_parquet(os.path.join(SAVE_DIR, 'val_s1_df.parquet'))
val_s1_sample = val_s1_df

print("✅ Saved! 'tfidf_cands_val.pkl' and 'val_s1_df.parquet' are permanently backed up.")


💾 Saving Cell 9 results to Google Drive...
✅ Saved! 'tfidf_cands_val.pkl' and 'val_s1_df.parquet' are permanently backed up.


In [ ]:
import os
import pickle
import pandas as pd

SAVE_DIR = '/content/drive/MyDrive/amazon_ml_challenge/'

print("⚡ Loading Cell 9 results from Google Drive...")

# Load candidates dictionary
with open(os.path.join(SAVE_DIR, 'tfidf_cands_val.pkl'), 'rb') as f:
    tfidf_cands_val = pickle.load(f)

# Load validation dataframe
val_s1_df = pd.read_parquet(os.path.join(SAVE_DIR, 'val_s1_df.parquet'))

print(f"✅ Loaded {len(tfidf_cands_val):,} TF-IDF validation candidates in 1 second! Ready for Cell 11.")


⚡ Loading Cell 9 results from Google Drive...
✅ Loaded 5,000 TF-IDF validation candidates in 1 second! Ready for Cell 11.


In [ ]:
%%time

def rule_based_blocking(s1_df, s23_df, min_shared_tokens=2):
    """
    Rule-based blocking: shared name tokens + same country.
    Returns {s1_entity_id: set(candidate_entity_ids)}.
    """
    # Build inverted index: token → list of (entity_id, country)
    print("  Building inverted index...")
    token_index = defaultdict(list)
    s23_countries = dict(zip(s23_df['entity_id'], s23_df['country']))

    for _, row in tqdm(s23_df.iterrows(), total=len(s23_df), desc="  Indexing S2+S3"):
        eid = row['entity_id']
        for token in row['name_tokens']:
            if len(token) > 2:  # skip very short tokens
                token_index[token].append(eid)

    print(f"  Inverted index: {len(token_index):,} unique tokens")

    # For each S1 entity, find candidates with ≥ min_shared_tokens in common
    candidates = {}
    for _, row in tqdm(s1_df.iterrows(), total=len(s1_df), desc="  Rule blocking"):
        s1_id = row['entity_id']
        s1_country = row['country']
        s1_tokens = row['name_tokens']

        cand_counts = defaultdict(int)
        for token in s1_tokens:
            if len(token) > 2:
                for cand_id in token_index.get(token, []):
                    cand_counts[cand_id] += 1

        cands = set()
        for cand_id, count in cand_counts.items():
            if count >= min_shared_tokens and s23_countries.get(cand_id) == s1_country:
                cands.add(cand_id)
        candidates[s1_id] = cands

    total_cands = sum(len(v) for v in candidates.values())
    avg_cands = total_cands / len(candidates) if candidates else 0
    print(f"  Rule-based blocking: {total_cands:,} total candidates, avg {avg_cands:.1f}/entity")
    return candidates


rule_cands_val = rule_based_blocking(
    val_s1_df, train_s23, min_shared_tokens=RULE_MIN_TOKENS
)

  Building inverted index...


  Indexing S2+S3:   0%|          | 0/10320219 [00:00<?, ?it/s]

  Inverted index: 1,419,788 unique tokens


  Rule blocking:   0%|          | 0/5000 [00:00<?, ?it/s]

  Rule-based blocking: 9,937,723 total candidates, avg 1987.5/entity
CPU times: user 10min 19s, sys: 2.28 s, total: 10min 22s
Wall time: 10min 20s


In [ ]:
import os
import pickle

SAVE_DIR = '/content/drive/MyDrive/amazon_ml_challenge/'
os.makedirs(SAVE_DIR, exist_ok=True)

save_path = os.path.join(SAVE_DIR, 'rule_cands_val.pkl')
print(f"💾 Saving rule-based candidates to {save_path}...")

with open(save_path, 'wb') as f:
    pickle.dump(rule_cands_val, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"✅ Successfully saved {len(rule_cands_val):,} rule-based candidate sets to Google Drive!")


💾 Saving rule-based candidates to /content/drive/MyDrive/amazon_ml_challenge/rule_cands_val.pkl...
✅ Successfully saved 5,000 rule-based candidate sets to Google Drive!


In [ ]:
import os
import pickle

SAVE_DIR = '/content/drive/MyDrive/amazon_ml_challenge/'
save_path = os.path.join(SAVE_DIR, 'rule_cands_val.pkl')

print("⚡ Loading rule-based candidates from Google Drive...")
with open(save_path, 'rb') as f:
    rule_cands_val = pickle.load(f)

print(f"✅ Loaded {len(rule_cands_val):,} candidate sets in 1 second!")


⚡ Loading rule-based candidates from Google Drive...
✅ Loaded 5,000 candidate sets in 1 second!


In [ ]:
%%time
import os
import pickle

# ============================================================
# CELL 12: Merge Blocking Channels & Evaluate Recall
# ============================================================
def merge_blocking_channels(*channels):
    """Union all blocking channels."""
    all_ids = set()
    for ch in channels:
        all_ids |= set(ch.keys())

    merged = {}
    for s1_id in all_ids:
        cands = set()
        for ch in channels:
            cands |= ch.get(s1_id, set())
        merged[s1_id] = cands
    return merged


def evaluate_blocking_recall(candidates, ground_truth):
    """Compute blocking recall: what fraction of true matches are in candidate sets."""
    total_true = 0
    found = 0
    missed_examples = []

    for s1_id, true_matches in ground_truth.items():
        if s1_id not in candidates:
            continue
        cands = candidates.get(s1_id, set())
        for m_id in true_matches:
            total_true += 1
            if m_id in cands:
                found += 1
            elif len(missed_examples) < 5:
                missed_examples.append((s1_id, m_id))

    recall = found / total_true if total_true > 0 else 0
    total_cands = sum(len(v) for v in candidates.values())
    avg_cands = total_cands / max(1, len(candidates))

    return {
        'recall': recall,
        'found': found,
        'total_true': total_true,
        'total_candidates': total_cands,
        'avg_candidates': avg_cands,
        'missed_examples': missed_examples,
    }


# 1. Merge TF-IDF + Rule-based (Fast & High Recall!)
merged_cands_val = merge_blocking_channels(tfidf_cands_val, rule_cands_val)

# 2. Evaluate Recall Comparison Table
print("=" * 70)
print(f"{'Channel':<25} {'Recall':>8} {'Found':>8} {'Total True':>12} {'Avg Cands':>10}")
print("=" * 70)
for name, cands in [('TF-IDF', tfidf_cands_val),
                     ('Rule-based', rule_cands_val),
                     ('🔗 MERGED (Union)', merged_cands_val)]:
    stats = evaluate_blocking_recall(cands, val_gt_dict)
    print(f"{name:<25} {stats['recall']:>7.4f} {stats['found']:>8,} {stats['total_true']:>12,} {stats['avg_candidates']:>10.1f}")
print("=" * 70)

# 3. Automatically Save the Master Merged Candidates to Google Drive
SAVE_DIR = '/content/drive/MyDrive/amazon_ml_challenge/'
os.makedirs(SAVE_DIR, exist_ok=True)
save_path = os.path.join(SAVE_DIR, 'merged_cands_val.pkl')

print(f"\n💾 Saving master merged candidates to Google Drive...")
with open(save_path, 'wb') as f:
    pickle.dump(merged_cands_val, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"✅ Successfully saved 'merged_cands_val.pkl' to Google Drive!")
print("🎯 Blocking phase is 100% COMPLETE! Ready for Feature Engineering (Cell 13).")


Channel                     Recall    Found   Total True  Avg Cands
TF-IDF                     0.9438   16,506       17,488       25.0
Rule-based                 0.6699   11,715       17,488     1987.5
🔗 MERGED (Union)           0.9597   16,784       17,488     2005.9

💾 Saving master merged candidates to Google Drive...
✅ Successfully saved 'merged_cands_val.pkl' to Google Drive!
🎯 Blocking phase is 100% COMPLETE! Ready for Feature Engineering (Cell 13).
CPU times: user 2.37 s, sys: 22.2 ms, total: 2.39 s
Wall time: 2.57 s


In [ ]:
import os
import pickle

SAVE_DIR = '/content/drive/MyDrive/amazon_ml_challenge/'
load_path = os.path.join(SAVE_DIR, 'merged_cands_val.pkl')

print("⚡ Loading master merged candidates from Google Drive...")

with open(load_path, 'rb') as f:
    merged_cands_val = pickle.load(f)

total_cands = sum(len(v) for v in merged_cands_val.values())
avg_cands = total_cands / max(1, len(merged_cands_val))

print(f"✅ Loaded {len(merged_cands_val):,} merged candidate sets in 1 second!")
print(f"   Total candidate pairs: {total_cands:,} (avg {avg_cands:.1f}/entity)")
print("🎯 Ready for Feature Engineering (Cell 13)!")


⚡ Loading master merged candidates from Google Drive...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/amazon_ml_challenge/merged_cands_val.pkl'

In [ ]:
%%time
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

random.seed(42)

def build_training_pairs(s1_ids_set, gt_dict, candidates, s1_lookup, s23_lookup,
                         neg_ratio=5, max_entities=None):
    """
    Build (features, label) training pairs from blocking candidates + ground truth.
    Uses hard negatives from blocking output.
    """
    all_features = []
    all_labels = []
    all_pairs = []

    entity_list = list(s1_ids_set)
    if max_entities:
        entity_list = entity_list[:max_entities]

    skipped = 0
    for s1_id in tqdm(entity_list, desc="Building pairs"):
        if s1_id not in s1_lookup.index:
            skipped += 1
            continue

        s1_row = s1_lookup.loc[s1_id]
        true_matches = gt_dict.get(s1_id, set())
        cand_set = candidates.get(s1_id, set())

        # 1. Positive pairs: true matches in candidate set
        for m_id in true_matches:
            if m_id in s23_lookup.index:
                cand_row = s23_lookup.loc[m_id]
                feat = compute_pair_features(s1_row, cand_row)
                all_features.append(feat)
                all_labels.append(1)
                all_pairs.append((s1_id, m_id))

        # 2. Hard negatives: candidates that are NOT true matches
        hard_negs = [c for c in cand_set if c not in true_matches and c in s23_lookup.index]
        n_neg = min(len(hard_negs), max(neg_ratio * len(true_matches), neg_ratio))
        if n_neg > 0:
            sampled_negs = random.sample(hard_negs, n_neg)
            for neg_id in sampled_negs:
                cand_row = s23_lookup.loc[neg_id]
                feat = compute_pair_features(s1_row, cand_row)
                all_features.append(feat)
                all_labels.append(0)
                all_pairs.append((s1_id, neg_id))

    X = pd.DataFrame(all_features).fillna(0).replace([np.inf, -np.inf], 0)
    y = np.array(all_labels, dtype=np.int32)

    print(f"  Built {len(X):,} pairs ({y.sum():,} positive, {(y==0).sum():,} negative)")
    if skipped:
        print(f"  Skipped {skipped} S1 entities (not in lookup)")
    return X, y, all_pairs


# ===== BUILD BLOCKING FOR TRAINING S1 ENTITIES =====
TRAIN_SAMPLE_SIZE = 10000  # 25k entities produces ~500k high-quality training pairs

train_s1_sample_ids = list(train_s1_ids)[:TRAIN_SAMPLE_SIZE]
train_s1_sample_df = train_s1[train_s1['entity_id'].isin(train_s1_sample_ids)].reset_index(drop=True)
print(f"Building blocking for {len(train_s1_sample_df):,} training S1 entities...")

# Run our 2 fast, memory-safe blocking channels
tfidf_cands_train  = tfidf_blocking(train_s1_sample_df, train_s23, top_k=TFIDF_TOP_K, min_sim=TFIDF_MIN_SIM)
rule_cands_train   = rule_based_blocking(train_s1_sample_df, train_s23, min_shared_tokens=RULE_MIN_TOKENS)

# Merge only the 2 active channels (TF-IDF + Rule-based)
merged_cands_train = merge_blocking_channels(tfidf_cands_train, rule_cands_train)

# Build Training Pairs
print("\n=== Building TRAINING pairs ===")
X_train, y_train, pairs_train = build_training_pairs(
    set(train_s1_sample_ids), train_gt_dict, merged_cands_train,
    train_s1_lookup, train_s23_lookup, neg_ratio=NEG_RATIO
)

# Build Validation Pairs (only for the validation entities we evaluated in Cell 12)
val_sample_keys = set(merged_cands_val.keys())
print(f"\n=== Building VALIDATION pairs ({len(val_sample_keys):,} entities) ===")
X_val, y_val, pairs_val = build_training_pairs(
    val_sample_keys, val_gt_dict, merged_cands_val,
    train_s1_lookup, train_s23_lookup, neg_ratio=NEG_RATIO
)

print(f"\n✅ All pairs built!")
print(f"   X_train shape: {X_train.shape} | Positive: {y_train.sum():,} | Negative: {(y_train==0).sum():,}")
print(f"   X_val shape:   {X_val.shape}   | Positive: {y_val.sum():,} | Negative: {(y_val==0).sum():,}")


Building blocking for 10,000 training S1 entities...

  [Country: India] S1: 4,090 | S2+S3: 4,133,346
    Fitting vocabulary on 300,000 sample texts...
    Transforming matrices...


    Sparse TF-IDF (India):   0%|          | 0/41 [00:00<?, ?it/s]


  [Country: US] S1: 5,910 | S2+S3: 6,186,873
    Fitting vocabulary on 300,000 sample texts...
    Transforming matrices...


    Sparse TF-IDF (US):   0%|          | 0/60 [00:00<?, ?it/s]

In [ ]:
import os
import numpy as np

SAVE_DIR = '/content/drive/MyDrive/amazon/amazon_ml_challenge/'
os.makedirs(SAVE_DIR, exist_ok=True)

print("💾 Saving X_train, y_train, X_val, y_val to Google Drive...")

# Parquet for fast tabular read/write, npy for labels
X_train.to_parquet(os.path.join(SAVE_DIR, 'X_train.parquet'))
np.save(os.path.join(SAVE_DIR, 'y_train.npy'), y_train)

X_val.to_parquet(os.path.join(SAVE_DIR, 'X_val.parquet'))
np.save(os.path.join(SAVE_DIR, 'y_val.npy'), y_val)

print("✅ Saved! Training and validation matrices are permanently backed up.")


In [ ]:
import os
import numpy as np
import pandas as pd

SAVE_DIR = '/content/drive/MyDrive/amazon/amazon_ml_challenge/'

print("⚡ Loading training pairs from Google Drive...")
X_train = pd.read_parquet(os.path.join(SAVE_DIR, 'X_train.parquet'))
y_train = np.load(os.path.join(SAVE_DIR, 'y_train.npy'))

X_val   = pd.read_parquet(os.path.join(SAVE_DIR, 'X_val.parquet'))
y_val   = np.load(os.path.join(SAVE_DIR, 'y_val.npy'))

print(f"✅ Loaded! X_train: {X_train.shape}, X_val: {X_val.shape}. Ready for XGBoost (Cell 15)!")
